In [8]:
# %%
# Classifier Diagnostics
# Covers:
#   1. Feature importance per cluster × model (RF + XGB)
#   2. Satellite/precip predictor dominance test
#   3. Class-conditional feature distributions (are classes separable?)
#   4. Routing contamination analysis — sample sizes and class purity after misrouting
#   5. Summary CSV exports and matplotlib figures

# %%
import os
import numpy as np
import pandas as pd
import joblib
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings("ignore")

# ─── Paths — adjust to your project layout ───────────────────────────────────
train_data_path   = r"..\Data\Processed\Classification_Regression\final_train_df.csv"
test_data_path    = r"..\Data\Processed\Classification_Regression\final_test_df.csv"
models_3class_dir = r"..\Data\Processed\Classification_Regression\Classification_models"
diag_out_dir      = r"..\Data\Processed\Classification_Regression\Diagnostics"
os.makedirs(diag_out_dir, exist_ok=True)

random_state_num = 64
target_variable  = "intensity_class"   # already present in the CSV; no recomputation needed

# Model filename pattern — edit suffix if your filenames differ
# Expected: cluster{c}_rf_classifier.pkl / cluster{c}_xgb_classifier.pkl   (3-class)
#           cluster{c}_rf_classifier_2classes.pkl                           (2-class, optional)
MODEL_SUFFIX_3   = "_classifier.pkl"
MODEL_SUFFIX_2   = "_classifier_2classes.pkl"

# ─── Feature groups — edit if your columns change ────────────────────────────
SATELLITE_PRECIP_COLS = [
    "era5L_prec_resampled", "era5L_prec_kriged",
    "har_prec", "persiann_prec", "chirps_prec"
]
ATMOS_COLS = [
    "era5L_wind_speed", "era5L_total_evaporation",
    "era5L_surface_pressure", "era5L_soil_moisture",
    "era5L_dewpoint_temperature"
]
LAND_SURFACE_COLS = [
    "modis_lst_d", "modis_lst_n", "modis_lst_diff", "modis_evi"
]
TERRAIN_COLS = [
    "elev_mean", "elev_std", "elev_max", "elev_min",
    "slope_mean", "aspect_mean", "elev_smoothed",
    "fetch_1000m_dist", "fetch_1000m_dir",
    "fetch_500m_dist", "fetch_500m_dir", "local_relief"
]
SPATIAL_COLS = ["x", "y", "spatial_ref_x", "spatial_ref_y"]

GROUP_MAP = {}
for c in SATELLITE_PRECIP_COLS: GROUP_MAP[c] = "Satellite/\nReanalysis Precip"
for c in ATMOS_COLS:             GROUP_MAP[c] = "Atmospheric\n(ERA5-L)"
for c in LAND_SURFACE_COLS:      GROUP_MAP[c] = "Land Surface\n(MODIS)"
for c in TERRAIN_COLS:           GROUP_MAP[c] = "Terrain /\nFetch"
for c in SPATIAL_COLS:           GROUP_MAP[c] = "Spatial\nCoordinates"

# ─── Colour palette ──────────────────────────────────────────────────────────
GROUP_COLORS = {
    "Satellite/\nReanalysis Precip": "#E07B54",
    "Atmospheric\n(ERA5-L)":        "#5B8DB8",
    "Land Surface\n(MODIS)":        "#6AAB6E",
    "Terrain /\nFetch":             "#9B74B7",
    "Spatial\nCoordinates":         "#B0B0B0",
    "Other":                        "#D0D0D0",
}
CLASS_COLORS = {0: "#6BAED6", 1: "#74C476", 2: "#E07B54"}
CLASS_NAMES  = {0: "No Rain", 1: "Light", 2: "Heavy"}
INTENSITY_LABELS = ["No Rain (0)", "Light (1)", "Heavy (2)"]

# ─── Load and prepare data ────────────────────────────────────────────────────
train_df = pd.read_csv(train_data_path)
test_df  = pd.read_csv(test_data_path)

# intensity_class is already in the CSV — just ensure correct dtype
train_df[target_variable] = train_df[target_variable].astype(int)
test_df[target_variable]  = test_df[target_variable].astype(int)

# Season encoding (keeps season_num as a usable feature if present downstream)
season_dict = {"winter": 1, "pre_monsoon": 2, "monsoon": 3, "post_monsoon": 4}
if "season" in train_df.columns:
    train_df["season_num"] = train_df["season"].map(season_dict)
    test_df["season_num"]  = test_df["season"].map(season_dict)

cluster_list = sorted(train_df["cluster"].unique().tolist())

# Drop non-feature columns — intensity_class is the target, insitu_prec is the raw
# measurement it was derived from (data leakage if kept), season is encoded above
cols_to_drop = [target_variable, "cluster", "date", "insitu_prec", "season"]
feature_cols = train_df.drop(columns=cols_to_drop, errors="ignore").columns.tolist()

print(f"Clusters      : {cluster_list}")
print(f"Features ({len(feature_cols)}): {feature_cols[:6]} ... (truncated)")
print(f"Target column : '{target_variable}'  |  classes: {sorted(train_df[target_variable].unique())}")

Clusters      : [0.0, 1.0, 2.0, 3.0]
Features (31): ['y', 'x', 'spatial_ref_x', 'era5L_prec_resampled', 'era5L_prec_kriged', 'har_prec'] ... (truncated)
Target column : 'intensity_class'  |  classes: [np.int64(0), np.int64(1)]


In [9]:
# %%
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1 — Feature importance per cluster × model
# ─────────────────────────────────────────────────────────────────────────────
importance_records = []

for cluster in cluster_list:
    for model_type in ["rf", "xgb"]:
        fname = f"cluster{cluster}_{model_type}{MODEL_SUFFIX_3}"
        fpath = os.path.join(models_3class_dir, fname)
        if not os.path.exists(fpath):
            print(f"  [SKIP] Not found: {fpath}")
            continue

        model = joblib.load(fpath)

        if model_type == "rf":
            importances = model.feature_importances_
        else:  # xgb — use gain-based importance
            imp_dict = model.get_booster().get_score(importance_type="gain")
            # XGBoost uses "f0","f1",... if feature names aren't set
            # Try to match by position
            try:
                importances = np.array([imp_dict.get(f, 0.0) for f in feature_cols])
            except Exception:
                importances = np.zeros(len(feature_cols))
                for k, v in imp_dict.items():
                    try:
                        idx = int(k.lstrip("f"))
                        if idx < len(importances):
                            importances[idx] = v
                    except ValueError:
                        pass
            # Normalise gain to sum = 1 for comparability with RF
            s = importances.sum()
            if s > 0:
                importances = importances / s

        for feat, imp in zip(feature_cols, importances):
            importance_records.append({
                "cluster":    cluster,
                "model":      model_type.upper(),
                "feature":    feat,
                "importance": imp,
                "group":      GROUP_MAP.get(feat, "Other"),
            })

imp_df = pd.DataFrame(importance_records)
imp_df.to_csv(os.path.join(diag_out_dir, "feature_importances.csv"), index=False)
print(f"\nFeature importance records: {len(imp_df)}")

# ─── Top-10 features per cluster×model ───────────────────────────────────────
print("\n===== Top-10 Features (averaged over clusters) =====")
top10_avg = (
    imp_df.groupby(["model", "feature"])["importance"]
    .mean()
    .reset_index()
    .sort_values(["model", "importance"], ascending=[True, False])
    .groupby("model")
    .head(10)
)
print(top10_avg.to_string(index=False))


Feature importance records: 248

===== Top-10 Features (averaged over clusters) =====
model                    feature  importance
   RF                 season_num    0.441627
   RF          era5L_prec_kriged    0.079795
   RF       era5L_prec_resampled    0.077742
   RF        era5L_soil_moisture    0.076560
   RF                   har_prec    0.052402
   RF era5L_dewpoint_temperature    0.042249
   RF              persiann_prec    0.028868
   RF                  modis_evi    0.024160
   RF    era5L_total_evaporation    0.023257
   RF     era5L_surface_pressure    0.018952
  XGB                aspect_mean    0.000000
  XGB                chirps_prec    0.000000
  XGB                   elev_max    0.000000
  XGB                  elev_mean    0.000000
  XGB                   elev_min    0.000000
  XGB              elev_smoothed    0.000000
  XGB                   elev_std    0.000000
  XGB era5L_dewpoint_temperature    0.000000
  XGB          era5L_prec_kriged    0.000000
  XGB       e

In [10]:
# %%
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2 — Satellite predictor dominance
# ─────────────────────────────────────────────────────────────────────────────
group_imp = (
    imp_df.groupby(["cluster", "model", "group"])["importance"]
    .sum()
    .reset_index()
    .rename(columns={"importance": "group_importance"})
)
group_imp.to_csv(os.path.join(diag_out_dir, "group_importances.csv"), index=False)
 
print("\n===== Group Importance (averaged across clusters) =====")
gi_avg = (
    group_imp.groupby(["model", "group"])["group_importance"]
    .mean()
    .reset_index()
    .sort_values(["model", "group_importance"], ascending=[True, False])
)
print(gi_avg.to_string(index=False))
 
sat_share = (
    group_imp[group_imp["group"] == "Satellite/\nReanalysis Precip"]
    .groupby("model")["group_importance"]
    .mean()
)
print(f"\n>>> Satellite/Reanalysis Precip mean importance share: RF={sat_share.get('RF', 0):.3f}  XGB={sat_share.get('XGB', 0):.3f}")
print("    (>0.60 would strongly support the dominant-predictor hypothesis)")



===== Group Importance (averaged across clusters) =====
model                         group  group_importance
   RF                         Other          0.441627
   RF Satellite/\nReanalysis Precip          0.250084
   RF         Atmospheric\n(ERA5-L)          0.175798
   RF         Land Surface\n(MODIS)          0.065013
   RF              Terrain /\nFetch          0.045466
   RF          Spatial\nCoordinates          0.011673
  XGB         Atmospheric\n(ERA5-L)          0.000000
  XGB         Land Surface\n(MODIS)          0.000000
  XGB                         Other          0.000000
  XGB Satellite/\nReanalysis Precip          0.000000
  XGB          Spatial\nCoordinates          0.000000
  XGB              Terrain /\nFetch          0.000000

>>> Satellite/Reanalysis Precip mean importance share: RF=0.250  XGB=0.000
    (>0.60 would strongly support the dominant-predictor hypothesis)


In [11]:
# %%
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3 — Class-conditional feature distributions
# Focus on satellite precip features: do No Rain / Light / Heavy overlap?
# ─────────────────────────────────────────────────────────────────────────────
print("\n===== Class-conditional stats for satellite precip features =====")
 
combined = pd.concat([train_df, test_df], ignore_index=True)
combined = combined.dropna(subset=[target_variable])
combined[target_variable] = combined[target_variable].astype(int)
 
overlap_records = []
for feat in SATELLITE_PRECIP_COLS:
    if feat not in combined.columns:
        continue
    for cls in [0, 1, 2]:
        vals = combined.loc[combined[target_variable] == cls, feat].dropna()
        overlap_records.append({
            "feature": feat,
            "class":   cls,
            "n":       len(vals),
            "median":  round(vals.median(), 4),
            "p25":     round(vals.quantile(0.25), 4),
            "p75":     round(vals.quantile(0.75), 4),
            "mean":    round(vals.mean(), 4),
            "std":     round(vals.std(), 4),
        })
 
overlap_df = pd.DataFrame(overlap_records)
overlap_df.to_csv(os.path.join(diag_out_dir, "class_conditional_stats.csv"), index=False)
print(overlap_df.to_string(index=False))


===== Class-conditional stats for satellite precip features =====
             feature  class      n  median    p25     p75    mean     std
era5L_prec_resampled      0 101921  0.8704 0.0379  4.5104  3.8998  8.4566
era5L_prec_resampled      1  16825 10.8663 5.2702 19.8286 17.5969 24.2331
era5L_prec_resampled      2      0     NaN    NaN     NaN     NaN     NaN
   era5L_prec_kriged      0 101921  0.8446 0.0358  4.4683  3.7735 32.4522
   era5L_prec_kriged      1  16825 10.8136 5.2229 19.7577 17.5382 24.3100
   era5L_prec_kriged      2      0     NaN    NaN     NaN     NaN     NaN
            har_prec      0 101921  0.0179 0.0000  2.8355  3.2839  8.6822
            har_prec      1  16825  7.8951 2.5999 17.4618 14.4190 21.0722
            har_prec      2      0     NaN    NaN     NaN     NaN     NaN
       persiann_prec      0 101921  0.1242 0.0000  2.1742  2.4929  5.8591
       persiann_prec      1  16825  4.1000 1.0762 10.4849  7.9344 10.6084
       persiann_prec      2      0     NaN   

In [12]:
# %%
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4 — Routing contamination
# For each cluster: load 3-class model, predict on test set, then measure:
#   (a) Effective sample size routed to each class-specific regressor
#   (b) Class purity (% of correctly routed samples)
#   (c) Contamination breakdown (what true classes ended up in each bin)
# ─────────────────────────────────────────────────────────────────────────────
routing_records  = []
purity_records   = []
contam_records   = []
 
for cluster in cluster_list:
    c_test = test_df[test_df["cluster"] == cluster].copy().dropna(subset=[target_variable])
    X_test = c_test.drop(columns=cols_to_drop, errors="ignore").values
    y_true = c_test[target_variable].values.astype(int)
 
    for model_type in ["rf", "xgb"]:
        fname = f"cluster{cluster}_{model_type}{MODEL_SUFFIX_3}"
        fpath = os.path.join(models_3class_dir, fname)
        if not os.path.exists(fpath):
            continue
 
        model   = joblib.load(fpath)
        y_pred  = model.predict(X_test)
 
        total_n = len(y_true)
 
        for pred_cls in [0, 1, 2]:
            routed_mask   = (y_pred == pred_cls)
            n_routed      = routed_mask.sum()
            true_in_bin   = y_true[routed_mask]
            n_correct     = (true_in_bin == pred_cls).sum()
            purity        = n_correct / n_routed if n_routed > 0 else np.nan
            pct_of_total  = n_routed / total_n
 
            routing_records.append({
                "cluster":         cluster,
                "model":           model_type.upper(),
                "predicted_class": pred_cls,
                "class_name":      CLASS_NAMES[pred_cls],
                "n_routed":        n_routed,
                "pct_of_test":     round(pct_of_total, 4),
                "n_correct":       n_correct,
                "purity":          round(purity, 4) if not np.isnan(purity) else np.nan,
            })
 
            # Contamination breakdown within this bin
            for true_cls in [0, 1, 2]:
                n_contam = (true_in_bin == true_cls).sum()
                contam_records.append({
                    "cluster":         cluster,
                    "model":           model_type.upper(),
                    "predicted_class": pred_cls,
                    "true_class":      true_cls,
                    "n":               n_contam,
                    "frac_of_bin":     round(n_contam / n_routed, 4) if n_routed > 0 else np.nan,
                })
 
        # Overall test accuracy (sanity check)
        acc = accuracy_score(y_true, y_pred)
        purity_records.append({
            "cluster":  cluster,
            "model":    model_type.upper(),
            "test_acc": round(acc, 4),
        })
 
routing_df = pd.DataFrame(routing_records)
contam_df  = pd.DataFrame(contam_records)
purity_df  = pd.DataFrame(purity_records)
 
routing_df.to_csv(os.path.join(diag_out_dir, "routing_summary.csv"),   index=False)
contam_df.to_csv(os.path.join(diag_out_dir, "contamination_detail.csv"), index=False)
 
print("\n===== Routing Summary =====")
print(routing_df.to_string(index=False))
print("\n===== Contamination Detail (fraction of routed bin that is each true class) =====")
print(contam_df.pivot_table(
    index=["cluster", "model", "predicted_class"],
    columns="true_class",
    values="frac_of_bin"
).to_string())

ValueError: X has 31 features, but RandomForestClassifier is expecting 32 features as input.

In [ ]:
# %%
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5 — Figures
# ─────────────────────────────────────────────────────────────────────────────
matplotlib.rcParams.update({
    "font.family": "sans-serif",
    "font.size":   10,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
 
# ── Figure 1: Feature importance heatmap (top 15 features, avg over clusters) ─
fig1, axes = plt.subplots(1, 2, figsize=(16, 7), sharey=True)
fig1.suptitle("Mean feature importance (averaged across clusters)", fontsize=13, y=1.01)
 
for ax, model_type in zip(axes, ["RF", "XGB"]):
    sub = imp_df[imp_df["model"] == model_type]
    feat_avg = (
        sub.groupby("feature")["importance"].mean()
        .sort_values(ascending=False)
        .head(15)
    )
    colors = [GROUP_COLORS.get(GROUP_MAP.get(f, "Other"), "#D0D0D0") for f in feat_avg.index]
    bars = ax.barh(feat_avg.index[::-1], feat_avg.values[::-1], color=colors[::-1], edgecolor="white", linewidth=0.5)
    ax.set_title(model_type, fontsize=11)
    ax.set_xlabel("Mean importance")
    ax.axvline(0, color="gray", linewidth=0.5)
    for bar, val in zip(bars, feat_avg.values[::-1]):
        ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height() / 2,
                f"{val:.3f}", va="center", fontsize=8, color="gray")
 
legend_patches = [Patch(facecolor=v, label=k.replace("\n", " ")) for k, v in GROUP_COLORS.items() if k != "Other"]
fig1.legend(handles=legend_patches, loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.04),
            frameon=False, fontsize=9)
fig1.tight_layout()
fig1.savefig(os.path.join(diag_out_dir, "fig1_feature_importance.png"), dpi=150, bbox_inches="tight")
print("\nFigure 1 saved.")
 
# ── Figure 2: Group importance stacked bar — per cluster × model ───────────────
fig2, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
fig2.suptitle("Feature group importance by cluster", fontsize=13)
 
for ax, model_type in zip(axes, ["RF", "XGB"]):
    sub = group_imp[group_imp["model"] == model_type].copy()
    pivot = sub.pivot_table(index="cluster", columns="group", values="group_importance", aggfunc="mean").fillna(0)
    pivot = pivot[[c for c in GROUP_COLORS if c in pivot.columns]]
    bottom = np.zeros(len(pivot))
    for grp in pivot.columns:
        ax.bar(
            [str(c) for c in pivot.index],
            pivot[grp].values,
            bottom=bottom,
            label=grp.replace("\n", " "),
            color=GROUP_COLORS[grp],
            edgecolor="white",
            linewidth=0.4,
        )
        bottom += pivot[grp].values
    ax.set_title(model_type, fontsize=11)
    ax.set_xlabel("Cluster")
    ax.set_ylabel("Cumulative importance")
    ax.set_ylim(0, 1)
    ax.axhline(0.6, color="red", linewidth=0.8, linestyle="--", alpha=0.6)
    ax.text(len(pivot) - 0.4, 0.61, "60% threshold", fontsize=7, color="red", alpha=0.8)
 
handles, labels = axes[0].get_legend_handles_labels()
fig2.legend(handles, labels, loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.06),
            frameon=False, fontsize=9)
fig2.tight_layout()
fig2.savefig(os.path.join(diag_out_dir, "fig2_group_importance.png"), dpi=150, bbox_inches="tight")
print("Figure 2 saved.")
 
# ── Figure 3: Class-conditional violin plots for satellite precip features ────
sat_feats_present = [f for f in SATELLITE_PRECIP_COLS if f in combined.columns]
n_feats = len(sat_feats_present)
fig3, axes = plt.subplots(1, n_feats, figsize=(4 * n_feats, 5), sharey=False)
fig3.suptitle("Class-conditional distributions: satellite/reanalysis precip features", fontsize=12)
 
if n_feats == 1:
    axes = [axes]
 
for ax, feat in zip(axes, sat_feats_present):
    data_per_class = [
        combined.loc[combined[target_variable] == cls, feat].dropna().values
        for cls in [0, 1, 2]
    ]
    parts = ax.violinplot(data_per_class, positions=[0, 1, 2], showmedians=True, showextrema=False)
    for pc, cls in zip(parts["bodies"], [0, 1, 2]):
        pc.set_facecolor(CLASS_COLORS[cls])
        pc.set_alpha(0.7)
    parts["cmedians"].set_color("black")
    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels(["No Rain", "Light", "Heavy"], fontsize=9)
    ax.set_title(feat.replace("era5L_", "").replace("_resampled", ""), fontsize=9)
    ax.set_ylabel("Value")
    ax.set_yscale("symlog")
 
fig3.tight_layout()
fig3.savefig(os.path.join(diag_out_dir, "fig3_class_conditional_distributions.png"), dpi=150, bbox_inches="tight")
print("Figure 3 saved.")
 
# ── Figure 4: Routing contamination heatmaps ─────────────────────────────────
fig4, axes = plt.subplots(len(cluster_list), 2, figsize=(12, 4 * len(cluster_list)))
fig4.suptitle("Routing bin purity: fraction of true class within each predicted bin\n"
              "(diagonal = pure routing; off-diagonal = contamination)", fontsize=11, y=1.01)
 
import matplotlib.colors as mcolors
cmap = plt.cm.RdYlGn
 
for row_i, cluster in enumerate(cluster_list):
    for col_i, model_type in enumerate(["RF", "XGB"]):
        ax = axes[row_i][col_i] if len(cluster_list) > 1 else axes[col_i]
        sub = contam_df[(contam_df["cluster"] == cluster) & (contam_df["model"] == model_type)]
        if sub.empty:
            ax.axis("off")
            continue
 
        mat = sub.pivot_table(index="predicted_class", columns="true_class", values="frac_of_bin").fillna(0)
        mat = mat.reindex(index=[0, 1, 2], columns=[0, 1, 2], fill_value=0)
 
        im = ax.imshow(mat.values, cmap=cmap, vmin=0, vmax=1, aspect="auto")
        ax.set_xticks([0, 1, 2]); ax.set_yticks([0, 1, 2])
        ax.set_xticklabels(["No Rain", "Light", "Heavy"], fontsize=8)
        ax.set_yticklabels(["No Rain", "Light", "Heavy"], fontsize=8)
        ax.set_xlabel("True class", fontsize=8)
        ax.set_ylabel("Predicted\n(routed to)", fontsize=8)
        ax.set_title(f"Cluster {cluster} | {model_type}", fontsize=9)
 
        # Annotate with fraction + n_routed
        routing_sub = routing_df[
            (routing_df["cluster"] == cluster) & (routing_df["model"] == model_type)
        ].set_index("predicted_class")
 
        for pred_i, pred_cls in enumerate([0, 1, 2]):
            n_routed = routing_sub.loc[pred_cls, "n_routed"] if pred_cls in routing_sub.index else 0
            for true_i, true_cls in enumerate([0, 1, 2]):
                val = mat.iloc[pred_i, true_i]
                color = "black" if val < 0.6 else "white"
                ax.text(true_i, pred_i, f"{val:.2f}", ha="center", va="center",
                        fontsize=8, color=color, fontweight="bold" if pred_i == true_i else "normal")
            # Annotate n on the right
            ax.text(3.05, pred_i, f"n={n_routed}", va="center", fontsize=7, color="gray",
                    transform=ax.get_yaxis_transform())
 
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
 
fig4.tight_layout()
fig4.savefig(os.path.join(diag_out_dir, "fig4_routing_contamination.png"), dpi=150, bbox_inches="tight")
print("Figure 4 saved.")

In [ ]:
# %%
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 6 — Summary diagnostic table
# ─────────────────────────────────────────────────────────────────────────────
summary_rows = []
for cluster in cluster_list:
    for model_type in ["RF", "XGB"]:
        r_sub = routing_df[(routing_df["cluster"] == cluster) & (routing_df["model"] == model_type)]
        if r_sub.empty:
            continue
        for _, row in r_sub.iterrows():
            sat_imp = group_imp[
                (group_imp["cluster"] == cluster) &
                (group_imp["model"] == model_type) &
                (group_imp["group"] == "Satellite/\nReanalysis Precip")
            ]["group_importance"].values
            summary_rows.append({
                "cluster":          cluster,
                "model":            model_type,
                "pred_class":       row["class_name"],
                "n_routed":         row["n_routed"],
                "pct_of_test":      row["pct_of_test"],
                "bin_purity":       row["purity"],
                "sat_precip_imp":   round(sat_imp[0], 4) if len(sat_imp) > 0 else np.nan,
            })
 
summary_table = pd.DataFrame(summary_rows)
summary_table.to_csv(os.path.join(diag_out_dir, "diagnostic_summary_table.csv"), index=False)
 
print("\n===== Diagnostic Summary Table =====")
print(summary_table.to_string(index=False))
print(f"\nAll outputs saved to: {diag_out_dir}")
print("Files written:")
for f in os.listdir(diag_out_dir):
    print(f"  {f}")